# 11.3 — Global parameter sensitivity

**Question.** Which historical configuration parameters explain outcomes when all seven vary jointly? The `test` profile validates four Latin-hypercube samples and artifact plumbing; it is intentionally too small for inferential importance. Screen/full profiles run the config-defined designs and require the env-resolved real feature table. Exact counts are printed before execution.

Signed artifacts resume under the legacy sensitivity root. Optimizer seeds are averaged before Spearman and random-forest permutation importance; held-out R² diagnoses model adequacy. Scikit-learn must be installed through the project `pipeline` or `all` extra. Importance is conditional on tested bounds and is not causal or an empirical calibration.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from estonia_landuse.sensitivity.analysis import rank_parameter_importance
from estonia_landuse.sensitivity.config import DEFAULT_SEEDS, GLOBAL_SAMPLE_COUNTS
from estonia_landuse.sensitivity.plots import plot_global_importance
from estonia_landuse.sensitivity.runner import run_manifest
from estonia_landuse.sensitivity.sampling import build_global_manifest, manifest_run_count, manifest_summary

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
HISTORICAL_ROOT = PROJECT_ROOT.parent.parent if PROJECT_ROOT.parent.name == ".worktrees" else PROJECT_ROOT
PROFILE = os.environ.get("SENSITIVITY_PROFILE", "test")
N_WORKERS = int(os.environ.get("SENSITIVITY_N_WORKERS", "2"))
OVERWRITE = os.environ.get("SENSITIVITY_OVERWRITE", "false").lower() == "true"
OUTPUT_ROOT = Path(os.environ.get("SENSITIVITY_OUTPUT_ROOT", PROJECT_ROOT / "data/processed/legacy_sensitivity")).resolve()
FEATURES_PATH = Path(os.environ.get("SENSITIVITY_FEATURES_PATH", HISTORICAL_ROOT / "data/processed/learned_carbon/features_with_forest.parquet")).resolve()
SEEDS = (0, 1) if PROFILE == "test" else DEFAULT_SEEDS[PROFILE]
N_SAMPLES = GLOBAL_SAMPLE_COUNTS[PROFILE]
OUTCOMES = ("biodiversity_gain", "carbon_gain", "cost", "changed_pct")


In [ ]:
if PROFILE == "test":
    position = np.linspace(0.0, 1.0, 12)
    context = pd.DataFrame({"cell_id": np.arange(1, 13), "forest_pct": 0.35 + 0.03 * position, "wetland_pct": 0.10 + 0.02 * position, "agriculture_pct": 0.30 - 0.03 * position, "grassland_pct": 0.15 - 0.02 * position, "urban_pct": np.full(12, 0.05), "water_pct": np.full(12, 0.05), "protected_overlap_pct": 0.05 * position, "wetland_suitability": 0.2 + 0.6 * position, "opportunity_cost_proxy": 0.1 + 0.5 * position, "predicted_tco2_ha_yr": 2.5 + 2.0 * position, "peat_overlap_pct": 0.4 * position})
    feature_columns = ["wetland_suitability", "opportunity_cost_proxy"]
else:
    if not FEATURES_PATH.exists(): raise FileNotFoundError(f"Missing historical feature input: {FEATURES_PATH}")
    context = pd.read_parquet(FEATURES_PATH)
    feature_columns = [name for name in ("urban_pct", "agriculture_pct", "grassland_pct", "forest_pct", "wetland_pct", "water_pct", "naturalness_score", "carbon_score", "protected_overlap_pct", "wetland_suitability", "biodiversity_proxy", "opportunity_cost_proxy", "rohemeeter_norm") if name in context]
    if not feature_columns: raise ValueError("No preserved Notebook 10 feature columns found")


In [ ]:
manifest = build_global_manifest(profile=PROFILE, n_samples=N_SAMPLES, sampler_seed=42, scenarios=("balanced",), seeds=SEEDS)
planned_runs = manifest_run_count(manifest)
manifest_summary(manifest)
display(manifest.head(6))


In [ ]:
statuses = run_manifest(context, feature_columns, manifest, OUTPUT_ROOT, PROFILE, overwrite=OVERWRITE, n_workers=min(N_WORKERS, planned_runs), progress=lambda completed, total, status: print(f"[{completed}/{total}] {status}"))
if statuses["status"].eq("failed").any(): raise RuntimeError(statuses.loc[statuses["status"].eq("failed"), ["sample_id", "seed", "error_message"]].to_string(index=False))
assert len(statuses) == planned_runs
display(statuses["status"].value_counts())


In [ ]:
metrics = pd.concat([pd.read_parquet(path) for path in statuses["metrics_path"]], ignore_index=True)
if metrics["sample_id"].nunique() < 8:
    print("Test profile completed; at least eight parameter samples are required for held-out importance.")
else:
    for outcome in OUTCOMES:
        ranked = rank_parameter_importance(metrics, outcome)
        print(f"{outcome}: held-out R² = {ranked.attrs['held_out_r2']:.3f}")
        display(ranked)
        figure, _ = plot_global_importance(ranked)
        display(figure)
        plt.close(figure)
